# 12. Multi-Target openFDA Drug Label Exploration

This notebook checks whether the multi-target drug candidates have FDA label information in openFDA.

Goal:
- Start from the multi-target ChEMBL drug recommendations.
- Query openFDA drug labels for each selected drug.
- Save raw API responses for reproducibility.
- Create processed CSV files that summarize label availability, indications, mechanisms, and warnings.

This notebook is part of the data preparation stage. It does not build the RAG app yet.

In [1]:
import json
import re
import time
from pathlib import Path
from urllib.parse import quote

import pandas as pd
import requests
from IPython.display import display

In [2]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw" / "openfda"
PROCESSED_DIR = DATA_DIR / "processed"

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

RECOMMENDATIONS_FILE = PROCESSED_DIR / "multi_target_drug_recommendations.csv"

print("Project root:", PROJECT_ROOT)
print("Raw folder:", RAW_DIR)
print("Processed folder:", PROCESSED_DIR)
print("Recommendations file:", RECOMMENDATIONS_FILE)

Project root: /Users/daniel/Documents/Projects/AI  Engineering Tools/datatalks/llm/ai-precision-medicine-lab/therapeutic-strategy-assistant
Raw folder: /Users/daniel/Documents/Projects/AI  Engineering Tools/datatalks/llm/ai-precision-medicine-lab/therapeutic-strategy-assistant/data/raw/openfda
Processed folder: /Users/daniel/Documents/Projects/AI  Engineering Tools/datatalks/llm/ai-precision-medicine-lab/therapeutic-strategy-assistant/data/processed
Recommendations file: /Users/daniel/Documents/Projects/AI  Engineering Tools/datatalks/llm/ai-precision-medicine-lab/therapeutic-strategy-assistant/data/processed/multi_target_drug_recommendations.csv


## 1. Load Multi-Target Drug Recommendations

The input file comes from notebook `09_multi_target_chembl_exploration.ipynb`.

In [3]:
if not RECOMMENDATIONS_FILE.exists():
    raise FileNotFoundError(
        f"Missing {RECOMMENDATIONS_FILE}. Run notebook 09_multi_target_chembl_exploration.ipynb first."
    )

recommendations_df = pd.read_csv(RECOMMENDATIONS_FILE)
print("Rows:", len(recommendations_df))
print("Columns:", list(recommendations_df.columns))
display(recommendations_df.head())

Rows: 207
Columns: ['target_symbol', 'target_display_name', 'target_full_name', 'target_chembl_id', 'target_pref_name', 'drug_name', 'molecule_chembl_id', 'molecule_type', 'action_type', 'mechanism_of_action', 'approval_status', 'max_phase', 'first_approval']


,target_symbol,target_display_name,target_full_name,target_chembl_id,target_pref_name,drug_name,molecule_chembl_id,molecule_type,action_type,mechanism_of_action,approval_status,max_phase,first_approval
0,ALK,ALK,ALK tyrosine kinase receptor,CHEMBL4247,ALK tyrosine kinase receptor,ALECTINIB HYDROCHLORIDE,CHEMBL3707320,Small molecule,INHIBITOR,ALK tyrosine kinase receptor inhibitor,Approved,4.0,2015.0
1,ALK,ALK,ALK tyrosine kinase receptor,CHEMBL4247,ALK tyrosine kinase receptor,ASP-3026,CHEMBL3545360,Small molecule,INHIBITOR,ALK tyrosine kinase receptor inhibitor,Investigational (Phase 1),1.0,NaN
2,ALK,ALK,ALK tyrosine kinase receptor,CHEMBL4247,ALK tyrosine kinase receptor,BRIGATINIB,CHEMBL3545311,Small molecule,INHIBITOR,ALK tyrosine kinase receptor inhibitor,Approved,4.0,2017.0
3,ALK,ALK,ALK tyrosine kinase receptor,CHEMBL4247,ALK tyrosine kinase receptor,CEP-37440,CHEMBL3951811,Small molecule,INHIBITOR,ALK tyrosine kinase receptor inhibitor,Investigational (Phase 1),1.0,NaN
4,ALK,ALK,ALK tyrosine kinase receptor,CHEMBL4247,ALK tyrosine kinase receptor,CERITINIB,CHEMBL2403108,Small molecule,INHIBITOR,ALK tyrosine kinase receptor inhibitor,Approved,4.0,2014.0


In [4]:
TARGET_ALIASES = {
    "EGFR": ["EGFR", "epidermal growth factor receptor"],
    "ERBB2": ["ERBB2", "HER2", "erb-b2", "erb b2", "human epidermal growth factor receptor 2"],
    "BRAF": ["BRAF", "B-RAF", "b raf"],
    "ALK": ["ALK", "anaplastic lymphoma kinase"],
    "KRAS": ["KRAS", "K-RAS", "k ras"],
    "VEGFA": ["VEGFA", "VEGF", "vascular endothelial growth factor"],
    "MET": ["MET", "HGFR", "hepatocyte growth factor receptor"],
    "PIK3CA": ["PIK3CA", "PI3K alpha", "phosphatidylinositol-4,5-bisphosphate 3-kinase catalytic subunit alpha"],
}

MAX_DRUGS_PER_TARGET = 8
OPENFDA_LABEL_URL = "https://api.fda.gov/drug/label.json"

print("Targets configured:", list(TARGET_ALIASES))
print("Max drugs per target:", MAX_DRUGS_PER_TARGET)

Targets configured: ['EGFR', 'ERBB2', 'BRAF', 'ALK', 'KRAS', 'VEGFA', 'MET', 'PIK3CA']
Max drugs per target: 8


## 2. Select Candidate Drugs for Label Search

To keep the API calls manageable, this notebook selects the strongest candidates per target first:
- FDA-approved / approved records first
- higher ChEMBL max phase next
- higher confidence score next, if available

In [5]:
def normalize_text(value):
    if pd.isna(value):
        return ""
    return re.sub(r"[^A-Z0-9]+", "", str(value).upper())


def clean_drug_name(value):
    """Make a drug name easier to search in openFDA."""
    if pd.isna(value):
        return ""
    value = str(value).strip()
    value = re.sub(r"\b(hydrochloride|sulfate|mesylate|maleate|phosphate|sodium|potassium|acetate)\b", "", value, flags=re.I)
    value = re.sub(r"\s+", " ", value).strip()
    return value


def approval_rank(value):
    text = str(value).lower()
    if "approved" in text:
        return 3
    if "phase 3" in text or "clinical" in text:
        return 2
    if "phase 2" in text or "phase 1" in text:
        return 1
    return 0


work_df = recommendations_df.copy()
work_df["clean_drug_name"] = work_df["drug_name"].apply(clean_drug_name)
work_df["normalised_drug_name"] = work_df["clean_drug_name"].apply(normalize_text)
work_df["approval_rank"] = work_df.get("approval_status", "").apply(approval_rank)

if "max_phase" not in work_df.columns:
    work_df["max_phase"] = 0
if "recommendation_score" not in work_df.columns:
    work_df["recommendation_score"] = 0

selected_df = (
    work_df.sort_values(
        ["target_symbol", "approval_rank", "max_phase", "recommendation_score"],
        ascending=[True, False, False, False],
    )
    .groupby("target_symbol", group_keys=False)
    .head(MAX_DRUGS_PER_TARGET)
    .reset_index(drop=True)
)

print("Selected rows:", len(selected_df))
display(selected_df[["target_symbol", "drug_name", "clean_drug_name", "approval_status", "max_phase"]].head(30))

Selected rows: 58


,target_symbol,drug_name,clean_drug_name,approval_status,max_phase
0,ALK,ALECTINIB HYDROCHLORIDE,ALECTINIB,Approved,4.0
1,ALK,BRIGATINIB,BRIGATINIB,Approved,4.0
2,ALK,CERITINIB,CERITINIB,Approved,4.0
3,ALK,CRIZOTINIB,CRIZOTINIB,Approved,4.0
4,ALK,ENSARTINIB,ENSARTINIB,Approved,4.0
5,ALK,ENTRECTINIB,ENTRECTINIB,Approved,4.0
6,ALK,LORLATINIB,LORLATINIB,Approved,4.0
7,ALK,ASP-3026,ASP-3026,Investigational (Phase 1),1.0
8,BRAF,DABRAFENIB MESYLATE,DABRAFENIB,Approved,4.0
9,BRAF,ENCORAFENIB,ENCORAFENIB,Approved,4.0


## 3. Query openFDA Labels

openFDA labels can be searched by generic name, brand name, or a broader text query. Not every ChEMBL molecule has a label record, especially research-only compounds.

In [6]:
def openfda_get(params, retries=3, pause=1.5):
    """GET from openFDA with retry handling. A 404 means no matching label was found."""
    for attempt in range(retries):
        try:
            response = requests.get(OPENFDA_LABEL_URL, params=params, timeout=(10, 60))
            if response.status_code == 200:
                return response.json()
            if response.status_code == 404:
                return {"results": []}
            print(f"  attempt {attempt + 1}: HTTP {response.status_code}; retrying")
        except requests.exceptions.RequestException as error:
            print(f"  attempt {attempt + 1}: {type(error).__name__}; retrying")
        time.sleep(pause)
    return {"results": []}


def build_openfda_queries(drug_name):
    clean_name = clean_drug_name(drug_name)
    if not clean_name:
        return []

    quoted = quote(clean_name)
    base = clean_name.split()[0]
    quoted_base = quote(base)

    queries = [
        f'openfda.generic_name:"{quoted}"',
        f'openfda.brand_name:"{quoted}"',
        f'openfda.substance_name:"{quoted}"',
    ]

    if base and base.lower() != clean_name.lower():
        queries.extend([
            f'openfda.generic_name:"{quoted_base}"',
            f'openfda.brand_name:"{quoted_base}"',
        ])

    queries.append(f'"{quoted}"')
    return list(dict.fromkeys(queries))


def flatten_text(value):
    if value is None:
        return ""
    if isinstance(value, list):
        return " ".join(str(item) for item in value if item is not None)
    return str(value)


def get_first(record, key):
    value = record.get(key)
    if isinstance(value, list):
        return " | ".join(str(item) for item in value[:3])
    if value is None:
        return ""
    return str(value)


def openfda_list(record, key):
    value = (record.get("openfda") or {}).get(key)
    if isinstance(value, list):
        return " | ".join(str(item) for item in value[:5])
    if value is None:
        return ""
    return str(value)


def label_mentions_target(record, target_symbol):
    aliases = TARGET_ALIASES.get(target_symbol, [target_symbol])
    searchable_fields = [
        "indications_and_usage",
        "clinical_pharmacology",
        "mechanism_of_action",
        "description",
        "warnings",
        "warnings_and_cautions",
    ]
    text = " ".join(flatten_text(record.get(field)) for field in searchable_fields).lower()
    return any(alias.lower() in text for alias in aliases)

In [7]:
raw_results = []
label_rows = []

for index, row in selected_df.iterrows():
    target_symbol = row["target_symbol"]
    drug_name = row["clean_drug_name"] or row["drug_name"]
    queries = build_openfda_queries(drug_name)

    print(f"[{index + 1}/{len(selected_df)}] {target_symbol} - {drug_name}")

    seen_set_ids = set()
    collected = []

    for query in queries:
        payload = openfda_get({"search": query, "limit": 5})
        results = payload.get("results", []) or []

        raw_results.append({
            "target_symbol": target_symbol,
            "drug_name": row["drug_name"],
            "molecule_chembl_id": row.get("molecule_chembl_id"),
            "query": query,
            "result_count": len(results),
            "response": payload,
        })

        for record in results:
            set_id = record.get("set_id") or record.get("id") or json.dumps(record, sort_keys=True)[:120]
            if set_id in seen_set_ids:
                continue
            seen_set_ids.add(set_id)
            collected.append((query, record))

        if collected:
            break

    for query, record in collected:
        label_rows.append({
            "target_symbol": target_symbol,
            "target_display_name": row.get("target_display_name", target_symbol),
            "target_full_name": row.get("target_full_name", ""),
            "drug_name": row["drug_name"],
            "clean_drug_name": drug_name,
            "molecule_chembl_id": row.get("molecule_chembl_id"),
            "approval_status": row.get("approval_status"),
            "max_phase": row.get("max_phase"),
            "query": query,
            "set_id": record.get("set_id"),
            "brand_name": openfda_list(record, "brand_name"),
            "generic_name": openfda_list(record, "generic_name"),
            "manufacturer_name": openfda_list(record, "manufacturer_name"),
            "route": openfda_list(record, "route"),
            "substance_name": openfda_list(record, "substance_name"),
            "indications_and_usage": get_first(record, "indications_and_usage"),
            "mechanism_of_action": get_first(record, "mechanism_of_action"),
            "clinical_pharmacology": get_first(record, "clinical_pharmacology"),
            "warnings": get_first(record, "warnings"),
            "warnings_and_cautions": get_first(record, "warnings_and_cautions"),
            "mentions_target_in_label": label_mentions_target(record, target_symbol),
            "source": "openFDA drug label",
        })

    time.sleep(0.25)

raw_file = RAW_DIR / "multi_target_openfda_labels_raw.json"
with raw_file.open("w") as f:
    json.dump(raw_results, f, indent=2)

labels_df = pd.DataFrame(label_rows)
print("Saved raw responses:", raw_file)
print("Label rows:", len(labels_df))
display(labels_df.head())

[1/58] ALK - ALECTINIB
[2/58] ALK - BRIGATINIB
[3/58] ALK - CERITINIB
[4/58] ALK - CRIZOTINIB
[5/58] ALK - ENSARTINIB
[6/58] ALK - ENTRECTINIB
[7/58] ALK - LORLATINIB
[8/58] ALK - ASP-3026
[9/58] BRAF - DABRAFENIB
[10/58] BRAF - ENCORAFENIB
[11/58] BRAF - REGORAFENIB
[12/58] BRAF - SORAFENIB TOSYLATE
[13/58] BRAF - VEMURAFENIB
[14/58] BRAF - CEP-32496
[15/58] BRAF - LIFIRAFENIB
[16/58] BRAF - PLIXORAFENIB
[17/58] EGFR - AFATINIB DIMALEATE
[18/58] EGFR - AMIVANTAMAB
[19/58] EGFR - BRIGATINIB
[20/58] EGFR - CETUXIMAB
[21/58] EGFR - DACOMITINIB
[22/58] EGFR - DACOMITINIB ANHYDROUS
[23/58] EGFR - ERLOTINIB
[24/58] EGFR - GEFITINIB
[25/58] ERBB2 - AFATINIB DIMALEATE
[26/58] ERBB2 - DACOMITINIB
[27/58] ERBB2 - DACOMITINIB ANHYDROUS
[28/58] ERBB2 - LAPATINIB DITOSYLATE
[29/58] ERBB2 - MARGETUXIMAB
[30/58] ERBB2 - MASOPROCOL
[31/58] ERBB2 - NERATINIB
[32/58] ERBB2 - NERATINIB
[33/58] KRAS - ADAGRASIB
[34/58] KRAS - SOTORASIB
[35/58] MET - AMIVANTAMAB
[36/58] MET - BEPERMINOGENE PERPLASMID
[37/

,target_symbol,target_display_name,target_full_name,drug_name,clean_drug_name,molecule_chembl_id,approval_status,max_phase,query,set_id,...,manufacturer_name,route,substance_name,indications_and_usage,mechanism_of_action,clinical_pharmacology,warnings,warnings_and_cautions,mentions_target_in_label,source
0,ALK,ALK,ALK tyrosine kinase receptor,ALECTINIB HYDROCHLORIDE,ALECTINIB,CHEMBL3707320,Approved,4.0,"openfda.generic_name:""ALECTINIB""",42c49deb-713b-427a-9670-08af08adcffb,...,"Genentech, Inc.",ORAL,ALECTINIB HYDROCHLORIDE,1 INDICATIONS AND USAGE ALECENSA is a kinase i...,12.1 Mechanism of Action Alectinib is a tyrosi...,12 CLINICAL PHARMACOLOGY 12.1 Mechanism of Act...,,5 WARNINGS AND PRECAUTIONS Hepatotoxicity: Mon...,True,openFDA drug label
1,ALK,ALK,ALK tyrosine kinase receptor,BRIGATINIB,BRIGATINIB,CHEMBL3545311,Approved,4.0,"openfda.generic_name:""BRIGATINIB""",0fe9ff20-d402-41f3-bc1e-7002ea7007db,...,"Takeda Pharmaceuticals America, Inc.",ORAL,BRIGATINIB,1 INDICATIONS AND USAGE ALUNBRIG is indicated ...,12.1 Mechanism of Action Brigatinib is a tyros...,12 CLINICAL PHARMACOLOGY 12.1 Mechanism of Act...,,5 WARNINGS AND PRECAUTIONS Interstitial Lung D...,True,openFDA drug label
2,ALK,ALK,ALK tyrosine kinase receptor,CERITINIB,CERITINIB,CHEMBL2403108,Approved,4.0,"openfda.generic_name:""CERITINIB""",fff5d805-4ffd-4e8e-8e63-6f129697563e,...,Novartis Pharmaceuticals Corporation,ORAL,CERITINIB,1 INDICATIONS AND USAGE ZYKADIA ® is indicated...,12.1 Mechanism of Action Ceritinib is a kinase...,12 CLINICAL PHARMACOLOGY 12.1 Mechanism of Act...,,5 WARNINGS AND PRECAUTIONS Gastrointestinal Ad...,True,openFDA drug label
3,ALK,ALK,ALK tyrosine kinase receptor,CRIZOTINIB,CRIZOTINIB,CHEMBL601719,Approved,4.0,"openfda.generic_name:""CRIZOTINIB""",2a51b0de-47d6-455e-a94c-d2c737b04ff7,...,Pfizer Laboratories Div Pfizer Inc,ORAL,CRIZOTINIB,1 INDICATIONS AND USAGE XALKORI is a kinase in...,12.1 Mechanism of Action Crizotinib is an inhi...,12 CLINICAL PHARMACOLOGY 12.1 Mechanism of Act...,,5 WARNINGS AND PRECAUTIONS • Hepatotoxicity: F...,True,openFDA drug label
4,ALK,ALK,ALK tyrosine kinase receptor,ENSARTINIB,ENSARTINIB,CHEMBL4113131,Approved,4.0,"openfda.generic_name:""ENSARTINIB""",1e1b2f79-678a-472a-b924-66909c8a4b2e,...,"Xcovery Holdings, Inc.",ORAL,ENSARTINIB HYDROCHLORIDE,1 INDICATIONS AND USAGE ENSACOVE is indicated ...,12.1 Mechanism of Action Ensartinib is a kinas...,12 CLINICAL PHARMACOLOGY 12.1 Mechanism of Act...,,5 WARNINGS AND PRECAUTIONS Interstitial Lung D...,True,openFDA drug label


## 4. Save Processed openFDA Tables

In [10]:
label_columns = [
    "target_symbol", "target_display_name", "target_full_name", "drug_name", "clean_drug_name",
    "molecule_chembl_id", "approval_status", "max_phase", "query", "set_id",
    "brand_name", "generic_name", "manufacturer_name", "route", "substance_name",
    "indications_and_usage", "mechanism_of_action", "clinical_pharmacology",
    "warnings", "warnings_and_cautions", "mentions_target_in_label", "source",
]

if labels_df.empty:
    labels_df = pd.DataFrame(columns=label_columns)
else:
    for column in label_columns:
        if column not in labels_df.columns:
            labels_df[column] = ""
    labels_df = labels_df[label_columns]

labels_file = PROCESSED_DIR / "multi_target_openfda_labels.csv"
labels_df.to_csv(labels_file, index=False)

print("Saved:", labels_file)
print("Rows:", len(labels_df))

Saved: /Users/daniel/Documents/Projects/AI  Engineering Tools/datatalks/llm/ai-precision-medicine-lab/therapeutic-strategy-assistant/data/processed/multi_target_openfda_labels.csv
Rows: 90


In [11]:
def safe_join(values, limit=5):
    cleaned = [str(value).strip() for value in values if pd.notna(value) and str(value).strip()]
    unique_values = list(dict.fromkeys(cleaned))
    return " | ".join(unique_values[:limit])


if labels_df.empty:
    summary_df = pd.DataFrame(columns=[
        "target_symbol", "drug_name", "molecule_chembl_id", "label_count", "has_openfda_label",
        "mentions_target_in_label", "brand_names", "generic_names", "top_indications", "source",
    ])
else:
    summary_df = labels_df.groupby(
        ["target_symbol", "drug_name", "molecule_chembl_id"], dropna=False
    ).agg(
        label_count=("set_id", "nunique"),
        mentions_target_in_label=("mentions_target_in_label", "max"),
        brand_names=("brand_name", safe_join),
        generic_names=("generic_name", safe_join),
        top_indications=("indications_and_usage", safe_join),
    ).reset_index()
    summary_df["has_openfda_label"] = summary_df["label_count"] > 0
    summary_df["source"] = "openFDA drug label"

summary_file = PROCESSED_DIR / "multi_target_openfda_summary.csv"
summary_df.to_csv(summary_file, index=False)

print("Saved:", summary_file)
print("Rows:", len(summary_df))
display(summary_df.head(20))

Saved: /Users/daniel/Documents/Projects/AI  Engineering Tools/datatalks/llm/ai-precision-medicine-lab/therapeutic-strategy-assistant/data/processed/multi_target_openfda_summary.csv
Rows: 48


,target_symbol,drug_name,molecule_chembl_id,label_count,mentions_target_in_label,brand_names,generic_names,top_indications,has_openfda_label,source
0,ALK,ALECTINIB HYDROCHLORIDE,CHEMBL3707320,1,True,ALECENSA,ALECTINIB HYDROCHLORIDE,1 INDICATIONS AND USAGE ALECENSA is a kinase i...,True,openFDA drug label
1,ALK,BRIGATINIB,CHEMBL3545311,1,True,Alunbrig,BRIGATINIB,1 INDICATIONS AND USAGE ALUNBRIG is indicated ...,True,openFDA drug label
2,ALK,CERITINIB,CHEMBL2403108,1,True,ZYKADIA,CERITINIB,1 INDICATIONS AND USAGE ZYKADIA ® is indicated...,True,openFDA drug label
3,ALK,CRIZOTINIB,CHEMBL601719,1,True,Xalkori,CRIZOTINIB,1 INDICATIONS AND USAGE XALKORI is a kinase in...,True,openFDA drug label
4,ALK,ENSARTINIB,CHEMBL4113131,1,True,ENSACOVE,ENSARTINIB,1 INDICATIONS AND USAGE ENSACOVE is indicated ...,True,openFDA drug label
5,ALK,ENTRECTINIB,CHEMBL1983268,1,True,Rozlytrek,ENTRECTINIB,1 INDICATIONS AND USAGE ROZLYTREK is a kinase ...,True,openFDA drug label
6,ALK,LORLATINIB,CHEMBL3286830,2,True,Lorbrena,LORLATINIB,1 INDICATIONS AND USAGE LORBRENA ® is indicate...,True,openFDA drug label
7,BRAF,DABRAFENIB MESYLATE,CHEMBL2105729,1,True,Tafinlar,DABRAFENIB,1 INDICATIONS AND USAGE TAFINLAR is a kinase i...,True,openFDA drug label
8,BRAF,ENCORAFENIB,CHEMBL3301612,1,True,BRAFTOVI,ENCORAFENIB,1 INDICATIONS AND USAGE BRAFTOVI is a kinase i...,True,openFDA drug label
9,BRAF,REGORAFENIB,CHEMBL1946170,1,True,Stivarga,REGORAFENIB,1 INDICATIONS AND USAGE STIVARGA is a kinase i...,True,openFDA drug label


In [12]:
selected_counts_df = selected_df.groupby("target_symbol").size().reset_index(name="selected_drug_count")

if summary_df.empty:
    found_counts_df = pd.DataFrame(columns=["target_symbol", "drugs_with_openfda_label", "drugs_where_label_mentions_target"])
else:
    found_counts_df = summary_df.groupby("target_symbol").agg(
        drugs_with_openfda_label=("has_openfda_label", "sum"),
        drugs_where_label_mentions_target=("mentions_target_in_label", "sum"),
    ).reset_index()

coverage_df = selected_counts_df.merge(found_counts_df, on="target_symbol", how="left")
coverage_df[["drugs_with_openfda_label", "drugs_where_label_mentions_target"]] = coverage_df[
    ["drugs_with_openfda_label", "drugs_where_label_mentions_target"]
].fillna(0).astype(int)
coverage_df["openfda_label_coverage_rate"] = (
    coverage_df["drugs_with_openfda_label"] / coverage_df["selected_drug_count"]
).round(3)
coverage_df["source_status"] = coverage_df["drugs_with_openfda_label"].apply(lambda value: "working" if value > 0 else "no labels found")

coverage_file = PROCESSED_DIR / "multi_target_openfda_coverage_summary.csv"
coverage_df.to_csv(coverage_file, index=False)

print("Saved:", coverage_file)
display(coverage_df)

Saved: /Users/daniel/Documents/Projects/AI  Engineering Tools/datatalks/llm/ai-precision-medicine-lab/therapeutic-strategy-assistant/data/processed/multi_target_openfda_coverage_summary.csv


,target_symbol,selected_drug_count,drugs_with_openfda_label,drugs_where_label_mentions_target,openfda_label_coverage_rate,source_status
0,ALK,8,7,7,0.875,working
1,BRAF,8,5,5,0.625,working
2,EGFR,8,8,8,1.000,working
3,ERBB2,8,8,7,1.000,working
4,KRAS,2,2,2,1.000,working
5,MET,8,7,7,0.875,working
6,PIK3CA,8,4,2,0.500,working
7,VEGFA,8,7,7,0.875,working


## 5. Final Check

After running this notebook, confirm that:
- raw openFDA responses were saved
- processed label rows were saved
- each target has a coverage row

In [13]:
print("openFDA Multi-Target Exploration Complete")
print("=" * 70)
print("Selected drug rows:", len(selected_df))
print("Label rows:", len(labels_df))
print("Summary rows:", len(summary_df))
print("Coverage rows:", len(coverage_df))
print("Files created:")
print("-", raw_file)
print("-", labels_file)
print("-", summary_file)
print("-", coverage_file)

display(coverage_df)

openFDA Multi-Target Exploration Complete
Selected drug rows: 58
Label rows: 90
Summary rows: 48
Coverage rows: 8
Files created:
- /Users/daniel/Documents/Projects/AI  Engineering Tools/datatalks/llm/ai-precision-medicine-lab/therapeutic-strategy-assistant/data/raw/openfda/multi_target_openfda_labels_raw.json
- /Users/daniel/Documents/Projects/AI  Engineering Tools/datatalks/llm/ai-precision-medicine-lab/therapeutic-strategy-assistant/data/processed/multi_target_openfda_labels.csv
- /Users/daniel/Documents/Projects/AI  Engineering Tools/datatalks/llm/ai-precision-medicine-lab/therapeutic-strategy-assistant/data/processed/multi_target_openfda_summary.csv
- /Users/daniel/Documents/Projects/AI  Engineering Tools/datatalks/llm/ai-precision-medicine-lab/therapeutic-strategy-assistant/data/processed/multi_target_openfda_coverage_summary.csv


,target_symbol,selected_drug_count,drugs_with_openfda_label,drugs_where_label_mentions_target,openfda_label_coverage_rate,source_status
0,ALK,8,7,7,0.875,working
1,BRAF,8,5,5,0.625,working
2,EGFR,8,8,8,1.000,working
3,ERBB2,8,8,7,1.000,working
4,KRAS,2,2,2,1.000,working
5,MET,8,7,7,0.875,working
6,PIK3CA,8,4,2,0.500,working
7,VEGFA,8,7,7,0.875,working
